In [1]:
# 주요 라이브러리 import
import warnings
warnings.filterwarnings(action='ignore')
import time
from IPython.display import Image
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import *
from sklearn.model_selection import cross_val_score
from sklearn.metrics import *
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn import datasets
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge, Lasso, ElasticNet

from matplotlib import rc, font_manager
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.DataFrame({
    "study_hours":  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],   # 하루 공부 시간
    "prev_score":   [50, 55, 60, 65, 70, 75, 80, 82, 85, 88],  # 지난 시험 점수
    "sleep_hours":  [5, 5, 6, 6, 7, 7, 7, 8, 8, 8],   # 수면 시간
    "game_hours":   [4, 4, 3, 3, 2, 2, 1, 1, 1, 0],   # 게임 시간
    "random_noise": [13, 8, 15, 3, 11, 2, 18, 7, 5, 9], # 점수와 거의 무관한 랜덤 값
    "score":        [50, 55, 60, 65, 70, 75, 80, 82, 88, 92]  # 이번 시험 점수(타깃)
})

print(df)

   study_hours  prev_score  sleep_hours  game_hours  random_noise  score
0            1          50            5           4            13     50
1            2          55            5           4             8     55
2            3          60            6           3            15     60
3            4          65            6           3             3     65
4            5          70            7           2            11     70
5            6          75            7           2             2     75
6            7          80            7           1            18     80
7            8          82            8           1             7     82
8            9          85            8           1             5     88
9           10          88            8           0             9     92


In [5]:
df.drop(columns=['score'])

,study_hours,prev_score,sleep_hours,game_hours,random_noise
0,1,50,5,4,13
1,2,55,5,4,8
2,3,60,6,3,15
3,4,65,6,3,3
4,5,70,7,2,11
5,6,75,7,2,2
6,7,80,7,1,18
7,8,82,8,1,7
8,9,85,8,1,5
9,10,88,8,0,9


In [6]:
# 문항과 답안을 분리
X = df.drop(columns=['score'])
y = df['score']

In [7]:
# 2. score와 각 설명 변수 간 상관계수 계산
corr_with_target = df.corr(numeric_only=True)["score"].drop("score")
print("score와의 상관계수:\n", corr_with_target)

score와의 상관계수:
 study_hours     0.998303
prev_score      0.996564
sleep_hours     0.963373
game_hours     -0.979227
random_noise   -0.208359
Name: score, dtype: float64


In [9]:
type(corr_with_target)

pandas.core.series.Series

In [8]:
corr_with_target.abs() #상관계수는 +/-여부 중요하지 않음. 숫자크기가 중요함.
#타입이 시리즈라서 왼쪽것이 index, 오른쪽것이 값임.
#가정 작은 값을 가지는 인덱스를 찾아서 제거해줄 예정임.

study_hours     0.998303
prev_score      0.996564
sleep_hours     0.963373
game_hours      0.979227
random_noise    0.208359
Name: score, dtype: float64

In [10]:
corr_with_target.abs().idxmin()
# 상관계수가 제일 작은 인덱스를 찾음.

'random_noise'

In [11]:
col_to_drop = corr_with_target.abs().idxmin()
col_to_drop

'random_noise'

In [12]:
X_reduced = X.drop(columns=[col_to_drop])

In [13]:
X_reduced.columns

Index(['study_hours', 'prev_score', 'sleep_hours', 'game_hours'], dtype='object')

In [14]:
# 5. 학습/테스트 데이터 나누기 (8:2)
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)
X_train.shape, y_train.shape

((8, 4), (8,))

In [15]:
# 6. 선형 회귀 모델 생성 및 학습
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [ ]:
# ai가 맞춘 답과 실제값의 차이정도를 나타내는 R2로 보통 정확도를 계산함.
# 검증기법으로 잔차의절대값합,잔차의제곱근합,잔차의제곱근합의루트,R2(크면 좋음)

In [16]:
# 7. R^2 점수 출력
print("\n[상관 약한 컬럼 제거 후]")
print("Train R^2 :", model.score(X_train, y_train))
print("Test  R^2 :", model.score(X_test, y_test))


[상관 약한 컬럼 제거 후]
Train R^2 : 0.9989594353202338
Test  R^2 : 0.9974916721780444


In [23]:
new_X = np.array([[10, 50, 5, 2], [2, 100, 3, 10]])
pred = model.predict(new_X)
pred

array([75.84782609, 75.41304348])

In [24]:
pred[0], pred[1]

(np.float64(75.84782608695653), np.float64(75.4130434782608))